In [2]:
import pandas as pd


In [3]:
df = pd.read_csv("IMDB Dataset.csv")

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [6]:
df.duplicated().sum()

np.int64(418)

In [7]:
df.shape

(50000, 2)

In [8]:
# 0 or 'index': Drop rows containing missing values (default).
# df.dropna(axis=0, inplace=True)

In [9]:
# Remove duplicate rows (keeps the first occurrence)
df = df.drop_duplicates()

In [10]:
df.shape

(49582, 2)

# Data Preprocessing

# Coverting every values into lowercase

In [11]:
# converting every character into lower case letter
df["review"] = df["review"].str.lower()

In [12]:
# Regular Expressions (Regex) are patterns used to search, match, replace, or manipulate text in a string.

# In Python, the re module provides functions to work with regular expressions.

In [13]:
    # http      -> match the word "http"
    # \S+       -> match one or more NON-space characters
    #             (keeps matching until the first space)
    # Example: "https://www.google.com today"
    #          matches -> "https://www.google.com"

In [14]:
import re
def remove_url(text):
    text = re.sub(r"http\S+", "", text) #This function uses regular expressions to identify URLs 
    return text                        # starting with http and removes them from each review in the DataFrame.
 

df["review"] = df["review"].apply(remove_url)

In [15]:
     # <      -> start of an HTML tag
    # .*?    -> match any characters (non-greedy, stops at the first '>')
    # >      -> end of the HTML tag
    # re.sub() replaces every matched HTML tag with an empty string.

In [16]:
def remove_html(text):
    text = re.sub(r"<.*?>", "", text)
    return text

df["review"] = df["review"].apply(remove_html)

In [17]:
#["A-Za-z0-9"] - its basically means keep this characters "A-Za-z0-9"] and remove all other things

In [18]:
def remove_punctutation(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "", text) 
    return text

df["review"] = df["review"].apply(remove_punctutation)

In [19]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production the filming tech...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


In [20]:
!pip install nltk

In [21]:
import nltk

In [22]:
nltk.download("punkt") #Punkt is used for tokenization, especially splitting text into sentences or words.
# "I love this product. It is amazing!"
# ["I love this product.", "It is amazing!"]
# ["I", "love", "this", "product"]

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [23]:
nltk.download("punkt_tab")
nltk.download("stopwords") #This downloads a list of common words that are often removed during NLP preprocessing. ex[the
# is,a,an,and,in,of,to]

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [24]:
from nltk.tokenize import word_tokenize #splits text into individual tokens.
from nltk.corpus import stopwords # # Provides a list of common stopwords

In [25]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")
    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")
    return text

df["review"] = df["review"].apply(remove_stopwords)

In [26]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti filming technique un...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


# Stemming


In [27]:
# played - play
# running - run

In [28]:
from nltk.stem import PorterStemmer # Imports PorterStemmer for reducing words to their root form

In [29]:
def stemming(text):
    ps = PorterStemmer()              # Create a stemmer object
    stemmed_words = []                # Empty list to store stemmed words
    
    tokens = word_tokenize(text)      # Split the text into individual tokens

    for token in tokens:
        stemmed_tokens = ps.stem(token)       # Convert each word to its stem
        stemmed_words.append(stemmed_tokens)  # Add the stemmed word to the list

    return " ".join(stemmed_words)    # Join the words back into one sentence
    
df["review"] = df["review"].apply(stemming)



In [30]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti film techniqu unssum oldtim...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


In [31]:
from sklearn.preprocessing import LabelEncoder # # Imports LabelEncoder to convert categorical labels into numbers
le = LabelEncoder()

In [32]:
df["sentiment"] = le.fit_transform(df["sentiment"])
y = df["sentiment"]

In [33]:
y.head(5)

0    1
1    1
2    1
3    0
4    1
Name: sentiment, dtype: int64

# Vectorization

In [34]:
# TF-IDF = Text → Numerical vectors.
# Common words get low weight, unique/important words get high weight.
# Example: "AI" > "love" because "love" appears in many documents.
#          ai    love  python
# 0  0.861037  0.5085  0.000000
# 1  0.000000  0.5085  0.861037
# 2  0.619130  0.4813  0.619130

In [35]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf = TfidfVectorizer(max_features=5000) #Keep only the 5000 most important words in the vocabulary.
X = tf.fit_transform(df["review"])


In [105]:
import joblib

joblib.dump(tf, "tfidf.pkl")

['tfidf.pkl']

In [36]:
print(type(y))
print(type(X))

<class 'pandas.core.series.Series'>
<class 'scipy.sparse._csr.csr_matrix'>


In [37]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=42, test_size=0.2
)

In [38]:
X_train.shape

(39665, 5000)

In [48]:
X_test.shape

(9917, 5000)

In [ ]:
X = X.toarray() # Convert the sparse TF-IDF matrix into a regular NumPy array

In [51]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [52]:
print(type(X_train))
print(type(y_test))

<class 'numpy.ndarray'>
<class 'pandas.core.series.Series'>


In [53]:
import torch
import torch.nn as nn

In [89]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32)

In [90]:
from torch.utils.data import TensorDataset, DataLoader
train_set = TensorDataset(X_train_tensor, y_train_tensor)
test_set = TensorDataset(X_test_tensor, y_test_tensor)

In [91]:
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=True)

# Build Our RNN

In [92]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.rnn = nn.RNN(input_size, hidden_size=128, num_layers=1, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1) #it return two thing out (which is all hidden state) and hn which is last hidden state

    def forward(self, x):
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep
        out,_ = self.rnn(x)

        # out[:, -1, :]
        # :   -> Take all reviews (all inputs in the batch)
        # -1  -> Take the last hidden state of each review
        # :   -> Take all hidden features (128 values)
        #
        # Pass the last hidden state to the Fully Connected layer
        # to get one prediction for each review.
        out = self.fc(out[:,-1,:])
        return out

In [93]:
#                 Hidden State 1   Hidden State 2   Hidden State 3
# Review 1              H1               H2               H3
# Review 2              H1               H2               H3
# Review 3              H1               H2               H3
# ...
# Review 64             H1               H2               H3

In [94]:
input_size = X_train.shape[1]

In [95]:
input_size

5000

In [96]:
import torch.optim as optim
model = RNN(input_size)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

In [97]:
torch.manual_seed(42)

In [98]:
epochs = 10
for epoch in range(epochs):
    model.train()
    for Xb, yb in train_loader:
        optimizer.zero_grad()
        Xb = Xb.unsqueeze(1)
        outputs = model(Xb)
        outputs = torch.sigmoid(outputs.squeeze())
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.36024078726768494
epoch = 2/10 and loss = 0.17635707557201385
epoch = 3/10 and loss = 0.22317901253700256
epoch = 4/10 and loss = 0.20479482412338257
epoch = 5/10 and loss = 0.4448501169681549
epoch = 6/10 and loss = 0.1019303947687149
epoch = 7/10 and loss = 0.20581486821174622
epoch = 8/10 and loss = 0.37554410099983215
epoch = 9/10 and loss = 0.3038690984249115
epoch = 10/10 and loss = 0.2279500812292099


In [104]:
torch.save(model.state_dict(), "rnn_model.pth") #Save your trained model

In [99]:
# Before sigmoid, your model's Linear layer can output any number:

# -5
#  2.7
#  10
# -1.3

# But for binary classification, we want something like:

# 0 to 1

# evaluation

In [103]:
model.eval()
with torch.no_grad():
    correct_vals = 0
    total_vals = 0
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)
        outputs = model(Xb)
        # Convert raw output into probability (0 to 1)
        # Then > 0.5 converts probability into True/False
        # .float() converts True/False into 1.0/0.0
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()
        total_vals += yb.size(0)
        # Compare predictions with actual labels
        # This gives True for correct and False for incorrect predictions
        # .sum() counts the True values (correct predictions)
        correct_vals += (predicted == yb).sum().item()
    print(f"accuracy = {correct_vals/total_vals * 100}")

accuracy = 85.70132096400121
